# Lab — Agent Loop

**Objective:** Run a bounded state machine with explicit plan/act/observe steps.

**Book track:** `08-agent-systems` · **Time:** 30–45 minutes · **Python:** 3.10+

Work through the cells in order. Each code cell should run top-to-bottom. Keep `main.py` in this folder aligned with your final answers—`pytest` validates that file.


## How to use this notebook

1. Open from the lab directory (`labs/04-agent-loop/`) in Jupyter, VS Code, or Codespaces.
2. Run cells sequentially; restart the kernel if you change earlier definitions.
3. Complete **Your turn** sections, then sync working code into `main.py`.
4. Run the verification cell (`pytest`) before you finish.


In [ ]:
from pathlib import Path

LAB_DIR = Path('.').resolve()
assert (LAB_DIR / 'main.py').exists(), (
    'Start Jupyter from the lab directory, e.g. labs/04-agent-loop/'
)
print('Lab directory:', LAB_DIR)


## Tasks

1. Diagram the state transitions for the default goal.
2. Add a step limit failure and verify graceful stop.
3. Insert one invalid action and define recovery behavior.
4. Log observations to a list you can inspect after the run.


## Step 1 — Agent state

Agents loop: **plan** an action, **act**, **observe** the result. Bound the loop with a step limit and explicit termination.


In [ ]:
from dataclasses import dataclass, field


@dataclass
class State:
    goal: str
    step: int = 0
    max_steps: int = 4
    observations: list[str] = field(default_factory=list)
    done: bool = False


State(goal='produce a verified draft')


## Step 2 — Planner and environment


In [ ]:
def plan(state: State) -> str:
    if not state.observations:
        return 'inspect requirements'
    if 'requirements inspected' in state.observations and 'draft created' not in state.observations:
        return 'create draft'
    return 'verify result'


def execute(action: str) -> str:
    outcomes = {
        'inspect requirements': 'requirements inspected',
        'create draft': 'draft created',
        'verify result': 'result verified',
    }
    return outcomes[action]


s = State(goal='demo')
print('first action:', plan(s))


## Step 3 — Run the bounded loop


In [ ]:
def run(goal: str, *, verbose: bool = True) -> State:
    state = State(goal=goal)
    while not state.done and state.step < state.max_steps:
        action = plan(state)
        observation = execute(action)
        state.observations.append(observation)
        state.step += 1
        state.done = observation == 'result verified'
        if verbose:
            print({'step': state.step, 'action': action, 'observation': observation})
    return state


final = run('produce a verified draft')
print('status:', 'complete' if final.done else 'step limit reached')
print('observations:', final.observations)


## Your turn

1. Sketch the state diagram for the default goal.
2. Lower `max_steps` and confirm graceful stop.
3. Add handling for an invalid action in `execute`.
4. Inspect `final.observations` after each run.

Copy the finished loop into `main.py`.


In [ ]:
# TODO: experiment with max_steps=2 and invalid actions
run('produce a verified draft', verbose=True)


## Verify

Run the test suite against `main.py` and `test_lab.py`.


In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'test_lab.py', '-q'],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr, file=sys.stderr)
assert result.returncode == 0, 'Tests failed—see output above'


## Reflection

- What broke first when you changed inputs?
- Which simpler baseline would you compare against in a design review?

## Extensions

- Add another case to `test_lab.py`.
- Link observations to a concept card on the AIEBOK site.
